In [1]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from stable_baselines3 import PPO
import matplotlib.pyplot as plt
from stable_baselines3.common.env_util import make_vec_env
import matplotlib.pyplot as plt
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import os
import torch

import os
import numpy as np
import ray
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.utils.framework import try_import_torch
import pathlib

In [2]:
torch, _ = try_import_torch()

SAVE_DIR = "./models/rllib_forex"
os.makedirs(SAVE_DIR, exist_ok=True)

1) Env (remplace par ton ForexEnv si tu veux)

In [3]:
class ForexEnv(gym.Env):
    """Environnement Forex simplifié pour Gymnasium / RL."""

    def __init__(self, config=None):
        super().__init__()
        self.initial_balance = 1000.0
        self.balance = self.initial_balance
        self.position_size = 1000.0
        self.max_steps = 50

        # Initialisation des prix sur 3 périodes
        self.prices = [1.1, 1.1, 1.1]
        self.current_step = 0

        # Actions : Buy, Sell, Hold
        self.action_space = gym.spaces.Discrete(3)
        # Observations : 3 derniers prix normalisés
        self.observation_space = gym.spaces.Box(low=0.0, high=10.0, shape=(3,), dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.balance = self.initial_balance
        self.prices = [1.1, 1.1, 1.1]
        self.current_step = 0
        return np.array(self.prices, dtype=np.float32), {}

    def step(self, action):
        old_price = self.prices[-1]
        new_price = float(old_price + np.random.normal(0, 0.01))
        self.prices.append(new_price)
        if len(self.prices) > 3:
            self.prices.pop(0)

        price_change_rel = (new_price - old_price) / old_price

        # Reward plus clair et amplifié
        if action == 0:  # Buy
            reward = price_change_rel * self.position_size * 500
        elif action == 1:  # Sell
            reward = -price_change_rel * self.position_size * 500
        else:  # Hold
            reward = -0.1  # petite pénalité pour inaction

        self.balance += reward
        self.current_step += 1

        # Déterminer terminated / truncated
        terminated = self.balance <= 0.0 or self.balance >= 2*self.initial_balance
        truncated = self.current_step >= self.max_steps

        obs = np.array(self.prices, dtype=np.float32)
        info = {"balance": self.balance}

        return obs, float(np.clip(reward, -1e3, 1e3)), terminated, truncated, info

2) Init Ray

In [4]:
ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False)

2025-10-19 10:49:38,659	INFO worker.py:2013 -- Started a local Ray instance.
d:\dev python\OrionTrader\.venv\lib\site-packages\ray\_private\worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.10.0
Ray version:,2.50.0


(SingleAgentEnvRunner pid=48816) DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!


3) Config de base (PPOConfig moderne)

In [5]:
base_config = (
    PPOConfig()
    .environment(env=ForexEnv)          # ton env
    .env_runners(num_env_runners=1)     # remplace rollouts()
    .framework("torch")
    .training(
        gamma=0.99,
        lr=3e-5,
        train_batch_size=4096,
        entropy_coeff=0.005,
        vf_loss_coeff=0.5,
        clip_param=0.15,
    )
    .resources(num_gpus=0)
    .debugging(log_level="INFO")
)

# Build initial algo
algo = base_config.build()
print("✅ PPO initialisé.")

2025-10-19 10:49:40,591	WARNING deprecation.py:50 -- DeprecationWarning: `build` has been deprecated. Use `AlgorithmConfig.build_algo` instead. This will raise an error in the future!
2025-10-19 10:49:40,593	WARNING algorithm_config.py:5050 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
d:\dev python\OrionTrader\.venv\lib\site-packages\ray\rllib\algorithms\algorithm.py:520: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
d:\dev python\OrionTrader\.v

✅ PPO initialisé.


4) Phases de fine-tuning

In [6]:
phases = [
    {"name": "A_explore", "timesteps": 100_000, "lr": 5e-5, "entropy_coeff": 0.01},
    {"name": "B_stabilize", "timesteps": 200_000, "lr": 3e-5, "entropy_coeff": 0.005},
    {"name": "C_exploit", "timesteps": 200_000, "lr": 1e-5, "entropy_coeff": 0.001},
]

5) Helper: robust parsing of result

In [7]:
def parse_result_for_steps_and_reward(result, algo_obj):
    """
    Retourne (steps_this_iter, reward_mean). Supporte plusieurs versions RLlib.
    """
    # 1️⃣ Steps
    step_keys = [
        "num_env_steps_trained",
        "num_env_steps_sampled",
        "timesteps_this_iter",
        "timesteps_total",
        "num_steps_trained",
    ]
    steps = None
    for k in step_keys:
        if k in result and isinstance(result[k], (int, float)):
            steps = int(result[k])
            break
    # nested fallback
    if steps is None:
        for container in ("env_runners", "env_runner", "info", "stats", "custom_metrics"):
            nested = result.get(container)
            if isinstance(nested, dict):
                for k in step_keys:
                    if k in nested and isinstance(nested[k], (int, float)):
                        steps = int(nested[k])
                        break
            if steps is not None:
                break
    if steps is None:
        steps = getattr(algo_obj, "num_env_steps_sampled", None) or getattr(algo_obj, "num_env_steps_trained", None) or 0
        try:
            steps = int(steps)
        except Exception:
            steps = 0

    # 2️⃣ Reward mean
    reward = None
    # Cas training
    for k in ("episode_reward_mean", "reward_mean", "policy_reward_mean"):
        if k in result:
            reward = result[k]
            break
    # Cas evaluation
    if reward is None and "evaluation" in result and "episode_reward_mean" in result["evaluation"]:
        reward = result["evaluation"]["episode_reward_mean"]
    # Nested fallback
    if reward is None:
        for container in ("env_runners", "env_runner", "info", "stats", "custom_metrics"):
            nested = result.get(container)
            if isinstance(nested, dict) and "episode_reward_mean" in nested:
                reward = nested["episode_reward_mean"]
                break

    # 3️⃣ Final fallback
    if reward is None:
        reward = 0.0

    return steps, reward


6) Sauvegardes multi-phase corrigées

tensorboard --logdir=logs/forex

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import pathlib

# Création du writer TensorBoard
writer = SummaryWriter(log_dir="logs/forex")

_total_global_steps = 0

for phase in phases:
    print(f"\n--- Starting phase {phase['name']} | timesteps={phase['timesteps']:,} "
          f"lr={phase['lr']} ent={phase['entropy_coeff']}")

    # Configuration de la phase
    phase_config = base_config.copy(copy_frozen=False)
    phase_config.environment(env=ForexEnv)
    phase_config.env_runners(num_env_runners=1)
    phase_config.training(
    lr=phase["lr"],
    entropy_coeff=phase["entropy_coeff"],
    train_batch_size=4000
    )
    
    # Rollout hyperparams
    phase_config.rollouts(
        rollout_fragment_length=200
    )
    
    new_algo = phase_config.build()

    # Transfert d'état
    try:
        state = algo.get_state()
        new_algo.set_state(state)
        print("State transferred via get_state()/set_state().")
    except Exception as e:
        try:
            w = algo.get_weights()
            new_algo.set_weights(w)
            print("State transferred via get_weights()/set_weights(). (fallback)")
        except Exception:
            print("⚠️ Warning: could not transfer state automatically:", e)

    total_steps_phase = 0

    while total_steps_phase < phase["timesteps"]:
        result = new_algo.train()
        print(result)  # pour debugger
        steps_this_iter = result.get("timesteps_this_iter", 0)
        if steps_this_iter == 0:
            print("⚠️ Aucun step produit, vérifie batch size et rollout_fragment_length !")
            break  # ou continue pour éviter boucle infinie
        steps_this_iter = result.get("timesteps_this_iter", 0)
        _total_global_steps += steps_this_iter
        total_steps_phase += steps_this_iter

        # Calculer reward moyen si disponible, sinon 0
        reward_mean = result.get("episode_reward_mean", 0.0)
        balance_mean = result.get("episode_balance_mean", 0.0)  # si balance est loggé dans custom metrics

        # Calculer balance moyenne si disponible
        episode_balances = result.get("hist_stats", {}).get("balance", [])
        balance_mean = sum(episode_balances) / len(episode_balances) if episode_balances else 0.0

        # Logging dans TensorBoard
        writer.add_scalar(f"{phase['name']}/reward_mean", reward_mean, _total_global_steps)
        writer.add_scalar(f"{phase['name']}/balance_mean", balance_mean, _total_global_steps)
        writer.add_scalar(f"{phase['name']}/steps", total_steps_phase, _total_global_steps)

        # Affichage console
        print(f"[{phase['name']}] phase_steps={total_steps_phase:,} (+{steps_this_iter}) "
              f"reward_mean={reward_mean:.4f} balance_mean={balance_mean:.2f}")

    # Sauvegarde checkpoint de la phase
    phase_dir = pathlib.Path(SAVE_DIR) / phase['name']
    phase_dir.mkdir(parents=True, exist_ok=True)
    ckpt = new_algo.save(str(phase_dir.resolve()))
    print(f"Checkpoint saved: {ckpt}")

    algo.stop()
    algo = new_algo

# Sauvegarde finale
final_dir = pathlib.Path(SAVE_DIR) / "final"
final_dir.mkdir(parents=True, exist_ok=True)
final_ckpt = algo.save_checkpoint(str(final_dir.resolve()))
print(f"✅ Full checkpoint saved at: {final_ckpt}")

# Fermer le writer TensorBoard
writer.close()


--- Starting phase A_explore | timesteps=100,000 lr=5e-05 ent=0.01


TypeError: AlgorithmConfig.training() got an unexpected keyword argument 'rollout_fragment_length'

7) Simple evaluation (local env episodes)

In [ ]:
from pathlib import Path
from ray.rllib.algorithms.ppo import PPO

# Chemin absolu vers le dossier contenant le checkpoint
checkpoint_dir = Path("models/rllib_forex/final").resolve()

# Configuration identique à l'entraînement
config = (
    PPO.get_default_config()
    .environment(ForexEnv)
    .framework("torch")  # ou "tf"
    .env_runners(num_env_runners=0)  # évaluation locale
    .evaluation(
        evaluation_duration=5,
        evaluation_num_env_runners=0,
        evaluation_config={"explore": False},
    )
)

# Création de l'algo
algo = PPO(config=config)

# ♻️ Restaurer le checkpoint depuis le dossier
algo.restore(str(checkpoint_dir))
print(f"✅ Checkpoint restauré depuis : {checkpoint_dir}")

# Évaluation
results = algo.evaluate()
mean_reward = results.get("evaluation", {}).get("episode_reward_mean", None)

print("\n=== Évaluation terminée ===")
print(f"🎯 Reward moyen : {mean_reward}")
